# Joint Reward Checkpoint Recipe v1

This runtime notebook converts the latest state-conditioned participation/drawdown failure into an auditable training/checkpoint objective. It is diagnostic only: no runtime promotion is allowed from this notebook.

## Experiment Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`
- **Direct comparison baseline**: `state_conditioned_participation_drawdown_objective_v1`
- **Role**: `diagnose`
- **Changed layer**: `reward`, `selection`, `model_selection`
- **Target window**: strict OOS plus `wf_2020`, `wf_2022`, `wf_2023_2024`
- **Acceptance gate**: every reviewed window must clear `relative_return >= +30pp` and `max_dd <= 15%`; no promotion from target alignment alone.
- **Hypothesis**: after path-level drawdown caps failed, the next useful lever is to expose the joint participation/drawdown trade-off as per-date training targets and checkpoint selection loss.
- **Expected trade-off**: better rebound participation may worsen drawdown unless downtrend cap violations are penalized in the same objective.
- **Failure mode**: if all candidate paths still fail the window gate, the recipe is only a PPO/checkpoint handoff, not a promoted runtime mechanism.
- **Comparison-table row key**: `joint_reward_checkpoint_recipe_v1`

## Historical Context

- Closest success: `state_objective_fold_native_teacher_router_v1` produced fold-local targets and selected `risk_budget_usage_v1` as the BC action surface.
- Closest failure: `state_conditioned_participation_drawdown_objective_v1` found no path-level equity drawdown controller that jointly cleared +30pp relative return, <=15% MaxDD, and rebound cluster participation.
- Why this is not a duplicate: this notebook does not rescreen path caps; it turns the failed path evidence into a reusable checkpoint/reward loss so the next PPO branch can retain by state-objective loss instead of legacy validation score.

## External Basis

- Qlib / workflow discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL / DRL trading environment separation: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfitting caution: https://doi.org/10.21314/JCF.2016.322
- Direct reinforcement learning / differential Sharpe lineage: https://papers.nips.cc/paper_files/paper/1998/hash/4e6cd95227cb0c280e99a195be5f6615-Abstract.html


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / 'src/soxl_quant').exists()), cwd)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveTrainingTargetConfig,
    build_state_objective_training_targets,
    score_state_objective_checkpoint_candidates,
    score_state_objective_target_alignment,
    summarize_state_objective_checkpoint_selection,
    summarize_state_objective_target_alignment,
    summarize_state_objective_training_targets,
)

CARD = ROOT / 'runtime/model_selection/card/joint_reward_checkpoint_recipe_v1'
SOURCE = ROOT / 'runtime/model_selection/card/state_conditioned_participation_drawdown_objective_v1'
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str) + '\n')

def read_json(path: Path) -> dict:
    return json.loads(path.read_text())


## Load Source Evidence

The source card already proved that post-hoc equity drawdown caps are not enough. This cell imports its selected teacher row and all candidate daily paths for target alignment scoring.

In [2]:
source_selection = read_json(SOURCE / 'selection.json')
source_summary = read_json(SOURCE / 'summary.json')
manifest = pd.read_csv(SOURCE / 'candidate_path_manifest.csv')
window_metrics = pd.read_csv(SOURCE / 'window_metrics.csv')
cluster_metrics = pd.read_csv(SOURCE / 'cluster_metrics.csv')
objective_scores = pd.read_csv(SOURCE / 'objective_scores.csv')

# The reusable target builder expects a generic cluster column. The latest source card
# writes cluster_id, so the notebook keeps this compatibility translation local.
if 'cluster' not in cluster_metrics.columns and 'cluster_id' in cluster_metrics.columns:
    cluster_metrics = cluster_metrics.rename(columns={'cluster_id': 'cluster'})

teacher_candidate = source_selection['selected_candidate']

candidate_paths: dict[str, dict[str, pd.DataFrame]] = {}
for row in manifest.itertuples(index=False):
    candidate_paths.setdefault(str(row.candidate), {})[str(row.window)] = pd.read_csv(ROOT / row.path)

teacher_daily_paths = candidate_paths[teacher_candidate]
print('teacher_candidate:', teacher_candidate)
print('candidates:', sorted(candidate_paths))
print('source_decision:', source_selection['decision'])
print('source_selected_objective_loss:', source_selection['selected_row']['objective_loss'])
manifest.head()


teacher_candidate: selected_router_no_equity_dd
candidates: ['selected_router_dd08_18_cap15', 'selected_router_dd10_20_cap20', 'selected_router_dd12_22_cap25', 'selected_router_no_equity_dd']
source_decision: diagnostic_no_objective_pass_no_promotion
source_selected_objective_loss: 967.395192410779


,candidate,window,path,rows
0,selected_router_no_equity_dd,strict_oos,runtime/model_selection/card/state_conditioned...,1605
1,selected_router_no_equity_dd,wf_2020,runtime/model_selection/card/state_conditioned...,252
2,selected_router_no_equity_dd,wf_2022,runtime/model_selection/card/state_conditioned...,250
3,selected_router_no_equity_dd,wf_2023_2024,runtime/model_selection/card/state_conditioned...,251
4,selected_router_dd12_22_cap25,strict_oos,runtime/model_selection/card/state_conditioned...,1605


## Build Per-Date Training Targets

The target table keeps separate loss terms for relative-return gap, drawdown gap, rebound participation, and downtrend risk caps. This avoids hiding the current failure behind a single scalar path rule.

In [3]:
config = StateObjectiveTrainingTargetConfig(
    target_relative_return=TARGET_RELATIVE_RETURN,
    target_max_dd=TARGET_MAX_DD,
    cluster_participation_floor=0.35,
    downtrend_position_cap=0.25,
    drawdown_state_threshold=-0.10,
)

targets = build_state_objective_training_targets(
    teacher_daily_paths,
    window_metrics=window_metrics,
    cluster_metrics=cluster_metrics,
    candidate=teacher_candidate,
    config=config,
)
target_summary = summarize_state_objective_training_targets(targets)
window_target_summary = pd.DataFrame(target_summary['window_summary'])

targets.to_csv(CARD / 'training_targets.csv', index=False)
window_target_summary.to_csv(CARD / 'window_target_summary.csv', index=False)
write_json(CARD / 'target_summary.json', target_summary)

print(json.dumps({
    'rows': target_summary['rows'],
    'top_weighted_window': target_summary['top_weighted_window'],
    'max_objective_weight': target_summary['max_objective_weight'],
    'rebound_rows': target_summary['rebound_rows'],
    'downtrend_rows': target_summary['downtrend_rows'],
    'required_terms': target_summary['required_terms'],
}, indent=2))
window_target_summary


{
  "rows": 2358,
  "top_weighted_window": "strict_oos",
  "max_objective_weight": 583.7075634676048,
  "rebound_rows": 281,
  "downtrend_rows": 1561,
  "required_terms": [
    "walkforward_relative_return_gap_penalty",
    "walkforward_drawdown_gap_penalty",
    "high_vol_rebound_participation_floor",
    "downtrend_drawdown_state_cap"
  ]
}


,window,rows,mean_objective_weight,max_objective_weight,rebound_rows,downtrend_rows,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,strict_oos,1605,370.653226,583.707563,215,1006,0.296688,0.309513,0.267823
1,wf_2020,252,100.758633,225.547199,66,118,0.352449,0.364980,0.264527
2,wf_2022,250,21.659558,22.254689,0,243,0.118209,0.118209,0.117184
3,wf_2023_2024,251,8.543194,10.759493,0,194,0.227884,0.227884,0.197738


## Score Candidate Alignment And Checkpoint Objective

This treats the current candidate paths as checkpoint-like alternatives. The selected row is not a promotion decision; it is the lowest-loss diagnostic reference for the next PPO/checkpoint-retention branch.

In [4]:
alignment_scores = score_state_objective_target_alignment(candidate_paths, targets, config=config)
alignment_summary = summarize_state_objective_target_alignment(alignment_scores)
checkpoint_scores = score_state_objective_checkpoint_candidates(alignment_scores)
checkpoint_summary = summarize_state_objective_checkpoint_selection(checkpoint_scores)

combined_scores = checkpoint_scores.merge(
    objective_scores,
    on='candidate',
    how='left',
    suffixes=('', '_source_objective'),
)

alignment_scores.to_csv(CARD / 'target_alignment_scores.csv', index=False)
checkpoint_scores.to_csv(CARD / 'checkpoint_objective_scores.csv', index=False)
combined_scores.to_csv(CARD / 'combined_candidate_scores.csv', index=False)
write_json(CARD / 'alignment_summary.json', alignment_summary)
write_json(CARD / 'checkpoint_summary.json', checkpoint_summary)

print('alignment_summary:', json.dumps(alignment_summary, indent=2))
print('checkpoint_summary:', json.dumps(checkpoint_summary, indent=2))
combined_scores[[
    'candidate',
    'checkpoint_objective_loss',
    'alignment_loss',
    'window_gate_loss',
    'rebound_position_mae',
    'risk_cap_violation',
    'window_gate',
    'coverage_gate',
    'objective_loss',
    'relative_loss',
    'drawdown_loss',
    'participation_loss',
]].head(10)


alignment_summary: {
  "decision": "target_alignment_no_promotion",
  "selected_candidate": "selected_router_no_equity_dd",
  "selected_alignment_loss": 5.987993900369647,
  "selected_window_gate": false,
  "selected_coverage_gate": true,
  "selected_missing_target_ratio": 0.0,
  "next_mechanism": "train_or_checkpoint_score_state_objective_targets",
  "required_terms": [
    "target_position_alignment_loss",
    "rebound_participation_alignment_loss",
    "downtrend_risk_cap_violation_loss",
    "walkforward_gate_gap_loss"
  ]
}
checkpoint_summary: {
  "decision": "checkpoint_objective_no_promotion",
  "selected_candidate": "selected_router_no_equity_dd",
  "selected_checkpoint_objective_loss": 11.760219169756816,
  "selected_checkpoint_selection_score": -11.760219169756816,
  "selected_window_gate": false,
  "selected_coverage_gate": true,
  "selected_alignment_loss": 5.987993900369647,
  "selected_window_gate_loss": 555.6456638404687,
  "selected_missing_target_ratio": 0.0,
  "next_m

,candidate,checkpoint_objective_loss,alignment_loss,window_gate_loss,rebound_position_mae,risk_cap_violation,window_gate,coverage_gate,objective_loss,relative_loss,drawdown_loss,participation_loss
0,selected_router_no_equity_dd,11.760219,5.987994,555.645664,0.098448,0.058660,False,True,967.395192,689.739584,267.011898,10.643711
1,selected_router_dd12_22_cap25,12.072924,6.148531,570.509080,0.111692,0.053805,False,True,984.291625,710.756855,262.595402,10.939368
2,selected_router_dd10_20_cap20,14.800502,7.535463,701.512711,0.167493,0.041209,False,True,1149.417935,891.267058,241.470244,16.680633
3,selected_router_dd08_18_cap15,18.406966,9.391436,870.669162,0.264280,0.022279,False,True,1391.042995,1110.092326,245.970897,34.979772


## Decision And Training Recipe

The recipe below is the artifact a future training run should consume conceptually: retain/checkpoint by state-objective loss, keep rebound participation and downtrend drawdown terms separate, and require full walk-forward validation before any promotion.

In [5]:
selected = checkpoint_scores.iloc[0].to_dict() if not checkpoint_scores.empty else {}
selected_candidate = str(selected.get('candidate', ''))
selected_source = objective_scores[objective_scores['candidate'].astype(str) == selected_candidate]
selected_source_row = selected_source.iloc[0].to_dict() if not selected_source.empty else {}

promotion_allowed = bool(checkpoint_summary.get('decision') == 'checkpoint_objective_promotion_review')
decision = checkpoint_summary['decision']
next_mechanism = checkpoint_summary['next_mechanism']

recipe = {
    'experiment': 'joint_reward_checkpoint_recipe_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_experiment': 'state_conditioned_participation_drawdown_objective_v1',
    'teacher_candidate': teacher_candidate,
    'selected_candidate': selected_candidate,
    'objective': {
        'target_relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'target_max_dd_max_pct': TARGET_MAX_DD,
        'checkpoint_ranking': 'checkpoint_objective_loss',
        'terms': [
            'target_position_alignment_loss',
            'walkforward_relative_return_gap_penalty',
            'walkforward_drawdown_gap_penalty',
            'high_vol_rebound_participation_floor',
            'downtrend_drawdown_state_cap',
            'checkpoint_objective_rebound_mae',
            'checkpoint_objective_risk_cap_violation',
        ],
    },
    'selected_metrics': {
        'checkpoint_objective_loss': float(selected.get('checkpoint_objective_loss', 0.0)),
        'alignment_loss': float(selected.get('alignment_loss', 0.0)),
        'window_gate_loss': float(selected.get('window_gate_loss', 0.0)),
        'rebound_position_mae': float(selected.get('rebound_position_mae', 0.0)),
        'risk_cap_violation': float(selected.get('risk_cap_violation', 0.0)),
        'max_relative_gap': float(selected.get('max_relative_gap', 0.0)),
        'max_drawdown_gap': float(selected.get('max_drawdown_gap', 0.0)),
        'source_objective_loss': float(selected_source_row.get('objective_loss', 0.0)),
        'source_relative_loss': float(selected_source_row.get('relative_loss', 0.0)),
        'source_drawdown_loss': float(selected_source_row.get('drawdown_loss', 0.0)),
        'source_participation_loss': float(selected_source_row.get('participation_loss', 0.0)),
    },
    'target_summary': {
        'rows': target_summary['rows'],
        'top_weighted_window': target_summary['top_weighted_window'],
        'max_objective_weight': target_summary['max_objective_weight'],
        'rebound_rows': target_summary['rebound_rows'],
        'downtrend_rows': target_summary['downtrend_rows'],
        'required_terms': target_summary['required_terms'],
    },
    'training_handoff': {
        'allowed_to_train_next': True,
        'allowed_to_promote_runtime': False,
        'warm_start_surface': 'risk_budget_usage_v1_or_compatible_state_objective_actor',
        'checkpoint_retention_metric': 'checkpoint_objective_loss',
        'must_validate': ['fixed_split', 'wf_2020', 'wf_2022', 'wf_2023_2024', 'strict_oos'],
    },
}

selection = {
    'experiment': 'joint_reward_checkpoint_recipe_v1',
    'created_at': recipe['created_at'],
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'state_conditioned_participation_drawdown_objective_v1',
    'role': 'diagnose',
    'changed_layer': ['reward', 'selection', 'model_selection'],
    'decision': decision,
    'next_mechanism': next_mechanism,
    'promotion_allowed': promotion_allowed,
    'training_allowed': True,
    'promotion_blocker': (
        'Candidate/checkpoint alternatives still fail the +30pp relative return and <=15% MaxDD window gate; '
        'use this only as a state-objective PPO/checkpoint-retention recipe.'
    ),
    'teacher_candidate': teacher_candidate,
    'selected_candidate': selected_candidate,
    'selected_checkpoint_objective_loss': recipe['selected_metrics']['checkpoint_objective_loss'],
    'selected_window_gate': bool(selected.get('window_gate', False)),
    'selected_coverage_gate': bool(selected.get('coverage_gate', False)),
    'selected_max_relative_gap': recipe['selected_metrics']['max_relative_gap'],
    'selected_max_drawdown_gap': recipe['selected_metrics']['max_drawdown_gap'],
    'required_terms': recipe['objective']['terms'],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://papers.nips.cc/paper_files/paper/1998/hash/4e6cd95227cb0c280e99a195be5f6615-Abstract.html',
    ],
    'acceptance_gate_for_next_run': {
        'relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'max_dd_max_pct': TARGET_MAX_DD,
        'fixed_split_required': True,
        'walkforward_required': True,
        'checkpoint_retention_metric_required': 'checkpoint_objective_loss',
        'no_runtime_promotion_from_recipe_only': True,
    },
    'sidecars': [
        'training_targets.csv',
        'window_target_summary.csv',
        'target_alignment_scores.csv',
        'checkpoint_objective_scores.csv',
        'combined_candidate_scores.csv',
        'target_summary.json',
        'alignment_summary.json',
        'checkpoint_summary.json',
        'training_objective_recipe.json',
        'summary.json',
    ],
}

summary = {
    'experiment': 'joint_reward_checkpoint_recipe_v1',
    'decision': decision,
    'selected_candidate': selected_candidate,
    'selected_checkpoint_objective_loss': recipe['selected_metrics']['checkpoint_objective_loss'],
    'selected_window_gate': selection['selected_window_gate'],
    'selected_coverage_gate': selection['selected_coverage_gate'],
    'target_rows': target_summary['rows'],
    'target_top_weighted_window': target_summary['top_weighted_window'],
    'target_rebound_rows': target_summary['rebound_rows'],
    'target_downtrend_rows': target_summary['downtrend_rows'],
    'conclusion': [
        'The latest path-level drawdown controller failure is now converted into per-date training targets and checkpoint objective scores.',
        'No candidate clears the window gate, so this is a training/checkpoint handoff only and not a runtime promotion.',
        'The next PPO branch should retain checkpoints by checkpoint_objective_loss and validate with fixed-split plus walk-forward windows before promotion.',
    ],
}

write_json(CARD / 'training_objective_recipe.json', recipe)
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)

print(json.dumps(summary, indent=2))


{
  "experiment": "joint_reward_checkpoint_recipe_v1",
  "decision": "checkpoint_objective_no_promotion",
  "selected_candidate": "selected_router_no_equity_dd",
  "selected_checkpoint_objective_loss": 11.760219169756816,
  "selected_window_gate": false,
  "selected_coverage_gate": true,
  "target_rows": 2358,
  "target_top_weighted_window": "strict_oos",
  "target_rebound_rows": 281,
  "target_downtrend_rows": 1561,
  "conclusion": [
    "The latest path-level drawdown controller failure is now converted into per-date training targets and checkpoint objective scores.",
    "No candidate clears the window gate, so this is a training/checkpoint handoff only and not a runtime promotion.",
    "The next PPO branch should retain checkpoints by checkpoint_objective_loss and validate with fixed-split plus walk-forward windows before promotion."
  ]
}
